#
[Setup Genie Space](https://docs.databricks.com/aws/en/genie/set-up)
[Setup Genie Space](https://docs.databricks.com/aws/en/genie/set-up)

In [1]:
import os
from dotenv import load_dotenv
#
load_dotenv()

#
# 1. Configure authentication credentials
# If running inside a Databricks Notebook, the host and token are automatically discovered.
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_PERSONAL_ACCESS_TOKEN")

In [6]:
import asyncio
import nest_asyncio
from langchain.agents import create_agent
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, DatabricksMCPServer, DatabricksMultiServerMCPClient
from langgraph.prebuilt import create_react_agent

nest_asyncio.apply()

workspace_client = WorkspaceClient()
host = workspace_client.config.host

databricks_mcp_client = DatabricksMultiServerMCPClient([
    DatabricksMCPServer(
        name="genie-space",
        url=f"{host}/api/2.0/mcp/genie/01f09dad81ae105b9cd77cec8660f37c",
        workspace_client=workspace_client,
    ),
])


# Create MCP tools from the configured servers
mcp_tools = asyncio.run(databricks_mcp_client.get_tools())

agent = create_agent(
    ChatDatabricks(endpoint="databricks-gemma-3-12b"),
    tools=mcp_tools,
)
# Create the agent graph with an LLM, tool set, and system prompt (if given)
#agent = create_tool_calling_agent(llm, mcp_tools, system_prompt)

result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Which product sold the most units ?"}]}
    )
print(result["messages"][-1].content)

The product that sold the most units is Golden Gate Ginger with 3,865 units sold.
